In [39]:
import pandas as pd

# Chargement des données
data = pd.read_csv('data/legislatives_2024/resultats.csv', sep=',', encoding='latin1')

data['LibNuaCand'] = data['LibNuaCand'].str.replace('\x88', 'è')
data['LibNuaCand'] = data['LibNuaCand'].str.replace('\x82', 'é')
data['LibNuaCand'] = data['LibNuaCand'].str.replace('\x87', 'ç')

data['RapportExprimes']


0        0,69
1       39,37
2       23,96
3       23,45
4        0,33
        ...  
4004     0,53
4005     3,28
4006    39,94
4007     1,37
4008     2,26
Name: RapportExprimes, Length: 4009, dtype: object

In [40]:
partis_uniques = data['LibNuaCand'].unique()
print(f'Partis politiques uniques présents dans les données :\n{partis_uniques}')

Partis politiques uniques présents dans les données :
['Extrème gauche' 'Rassemblement National' 'Les Républicains'
 'Union de la gauche' 'Droite souverainiste'
 'Ensemble ! (Majorité présidentielle)' 'Extrème droite' 'Divers'
 'Ecologistes' 'Divers droite' 'Reconquète !' "Union de l'extrème droite"
 'Divers gauche' 'Union des Démocrates et Indépendants' 'Régionaliste'
 'Divers centre' 'Horizons' 'Parti communiste français' 'Parti socialiste'
 'La France insoumise' 'Les Ecologistes' 'Parti radical de gauche']


In [42]:
parti_couleurs = {
    'Rassemblement National': 'navy',
    'Les Républicains': 'navy',
    'Ensemble ! (Majorité présidentielle)': 'darkorange',
    'Ecologistes': 'green',
    'Les Ecologistes': 'green',
    'La France insoumise': 'red',
    'Parti socialiste': 'red',
    'Extrème gauche': 'lightgrey',
    'Union de la gauche': 'lightgrey',
    'Droite souverainiste': 'lightgrey',
    'Extrème droite': 'lightgrey',
    'Divers': 'lightgrey',
    'Divers droite': 'lightgrey',
    'Reconquète !': 'lightgrey',
    "Union de l'extrème droite": 'lightgrey',
    'Divers gauche': 'lightgrey',
    'Union des Démocrates et Indépendants': 'lightgrey',
    'Régionaliste': 'lightgrey',
    'Divers centre': 'lightgrey',
    'Horizons': 'lightgrey',
    'Parti communiste français': 'lightgrey',
    'Parti radical de gauche': 'lightgrey'
}

data['Couleur'] = data['LibNuaCand'].map(parti_couleurs)


In [43]:
# Groupement par circonscription et sélection du candidat avec le plus grand RapportExprimes
resultats_max = data.loc[data.groupby('CodCirElec')['RapportExprimes'].idxmax()]

len(resultats_max)

577

In [50]:
import pandas as pd
import json
import plotly.express as px
import re

# Charger les résultats des élections avec encodage spécifié
file_path = 'data/legislatives_2024/resultats.csv'

# Essayer plusieurs encodages pour trouver celui qui fonctionne
encodings = ['utf-8', 'latin1', 'ISO-8859-1']
for enc in encodings:
    try:
        data = pd.read_csv(file_path, encoding=enc)
        print(f'Successfully loaded with encoding: {enc}')
        break
    except UnicodeDecodeError:
        print(f'Failed to load with encoding: {enc}')
        continue

# Remplacer les caractères incorrects
data['LibNuaCand'] = data['LibNuaCand'].str.replace('\x88', 'è')
data['LibNuaCand'] = data['LibNuaCand'].str.replace('\x82', 'é')
data['LibNuaCand'] = data['LibNuaCand'].str.replace('\x87', 'ç')

# Associer les couleurs et renommer les partis en rouge en "NFP"
parti_couleurs = {
    'Rassemblement National': 'navy',
    'Les Républicains': 'navy',
    'Ensemble ! (Majorité présidentielle)': 'darkorange',
    'NFP': 'red',
    'Extrème gauche': 'lightgrey',
    'Union de la gauche': 'lightgrey',
    'Droite souverainiste': 'lightgrey',
    'Extrème droite': 'lightgrey',
    'Divers': 'lightgrey',
    'Divers droite': 'lightgrey',
    'Reconquète !': 'lightgrey',
    "Union de l'extrème droite": 'lightgrey',
    'Divers gauche': 'lightgrey',
    'Union des Démocrates et Indépendants': 'lightgrey',
    'Régionaliste': 'lightgrey',
    'Divers centre': 'lightgrey',
    'Horizons': 'lightgrey',
    'Parti communiste français': 'lightgrey',
    'Parti radical de gauche': 'lightgrey'
}

data['Couleur'] = data['LibNuaCand'].map(parti_couleurs)
data['LibNuaCand'] = data['LibNuaCand'].replace(
    {
        'La France insoumise': 'NFP',
        'Parti socialiste': 'NFP',
        'Ecologistes': 'NFP',
        'Les Ecologistes': 'NFP'
    }
)

# Garder le candidat ayant le plus grand RapportExprimes pour chaque circonscription
data = data.loc[data.groupby('LibCirElec')['RapportExprimes'].idxmax()]

# Extraire le numéro de la circonscription de 'LibCirElec'
data['num_circ'] = data['LibCirElec'].apply(lambda x: re.findall(r'\d+', x)[0])

# Charger le fichier GeoJSON des circonscriptions
with open('data/legislatives_2024/circonscriptions-legislatives.json') as f:
    geojson = json.load(f)

# Créer un DataFrame pour les circonscriptions avec leurs couleurs
circonscriptions = pd.DataFrame([
    {
        "num_circ": row['num_circ'],
        "LibNuaCand": row['LibNuaCand'],
        "Couleur": row['Couleur']
    }
    for index, row in data.iterrows()
])

# Convertir le numéro de circonscription en chaîne de caractères
circonscriptions['num_circ'] = circonscriptions['num_circ'].astype(str)

# Créer une carte choroplèthe avec Plotly
fig = px.choropleth_mapbox(
    circonscriptions,
    geojson=geojson,
    locations='num_circ',
    featureidkey='properties.num_circ',
    color='LibNuaCand',
    color_discrete_map=parti_couleurs,
    hover_name='LibNuaCand',
    mapbox_style="carto-positron",
    zoom=5,
    center={"lat": 46.603354, "lon": 1.888334},
    opacity=0.5
)

# Mettre à jour la mise en page de la carte
fig.update_layout(
    title_text='Résultats des élections législatives 2024 - Premier tour',
    legend_title_text='Partis politiques',
    legend=dict(
        itemsizing='constant',
        bgcolor="rgba(255, 255, 255, 0.7)"
    )
)

# Afficher la carte
fig.show()


Failed to load with encoding: utf-8
Successfully loaded with encoding: latin1


d:\Programmes\Python\Lib\site-packages\plotly\express\_core.py:2065: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



In [51]:
circonscriptions

,LibCirElec,LibNuaCand,Couleur
0,10me circonscription,Les Républicains,navy
1,11me circonscription,Les Républicains,navy
2,12me circonscription,Rassemblement National,navy
3,13me circonscription,Les Républicains,navy
4,14me circonscription,Les Républicains,navy
5,15me circonscription,Rassemblement National,navy
6,16me circonscription,Rassemblement National,navy
7,17me circonscription,Rassemblement National,navy
8,18me circonscription,Rassemblement National,navy
9,19me circonscription,Rassemblement National,navy
